# Silver 재전송·오류·충돌·재시도 검증

- **추가:** 고정 입력과 독립 기대값으로 Spark/Delta 저장을 확인합니다.
- **추가:** 운영 테이블 대신 commerce_poc.silver_poc_tests의 실행별 테이블을 사용합니다.

일반 컴퓨트 DBR 16.1 이상에서 실행합니다. 합성 출처·시각이므로 지연 측정 자료로 사용하지 않습니다. 아직 원격 실행하지 않았습니다.

## 1. 저장소 코드와 매핑 연결

In [ ]:
import json
import sys
from pathlib import Path

dbutils.widgets.text("project_root", "")
configured_root = dbutils.widgets.get("project_root").strip()
if configured_root:
    ROOT = Path(configured_root)
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "01_contracts/data/reference/sku_mapping.json").is_file()), None)
if ROOT is None or not (ROOT / "02_pipeline/silver/channel_a.py").is_file():
    raise ValueError("project_root에 Databricks Git 폴더의 저장소 루트(/Workspace/...)를 입력하세요")

sys.path.insert(0, str(ROOT / "02_pipeline/silver"))
import silver_pipeline as pipeline
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
MAPPING = json.loads((ROOT / "01_contracts/data/reference/sku_mapping.json").read_text(encoding="utf-8"))
print({"project_root": str(ROOT), "sku_mapping_count": len(MAPPING)})


## 2. 격리된 입력·테이블 생성

- 새 실행 ID로 테이블과 체크포인트를 분리하고 기존 데이터를 보존합니다.
- HTTP에서 거절되는 malformed JSON은 이 시험의 Bronze 형식 DataFrame으로 주입합니다.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

FIXTURES = ROOT / "01_contracts/data/fixtures/channel_a"
RUN_ID = uuid4().hex[:12]
SCHEMA = "commerce_poc.silver_poc_tests"
BRONZE = f"{SCHEMA}.bronze_{RUN_ID}"
SILVER = f"{SCHEMA}.events_{RUN_ID}"
QUARANTINE = f"{SCHEMA}.quarantine_{RUN_ID}"
CHECKPOINT = f"/Volumes/commerce_poc/bronze/checkpoints/silver_validation/{RUN_ID}"
EXPECTED = json.loads((FIXTURES / "quality_expected.json").read_text(encoding="utf-8"))
ERROR_CASES = json.loads((FIXTURES / "quality_cases.json").read_text(encoding="utf-8"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
pipeline.create_tables(spark, SILVER, QUARANTINE)

def bronze_rows(raw_payloads, offset_start=0):
    instant = datetime(2026, 10, 12, tzinfo=timezone.utc)
    rows = [(raw, "fixture-event-hub", 0, offset_start + i, instant, instant)
            for i, raw in enumerate(raw_payloads)]
    return spark.createDataFrame(rows, pipeline.SOURCE_SCHEMA)

def counts():
    return {"silver": spark.table(SILVER).count(), "quarantine": spark.table(QUARANTINE).count()}

replay = (FIXTURES / "replay_events.jsonl").read_text(encoding="utf-8").splitlines()
bronze_rows(replay).write.format("delta").mode("errorifexists").saveAsTable(BRONZE)
print({"run_id": RUN_ID, "bronze": BRONZE, "silver": SILVER, "quarantine": QUARANTINE})


## 3. v1/v2 12건 → 정상 이벤트 6개

- Delta 스트리밍 입력을 foreachBatch로 처리하고 종료를 기다립니다.
- 60초 뒤에도 실행 중이면 상태/예외를 확인합니다. 기다림이 끝났다는 이유로 성공 처리하지 않습니다.

In [ ]:
validation_query = pipeline.start_stream(
    spark, bronze_table=BRONZE, silver_table=SILVER, quarantine_table=QUARANTINE,
    checkpoint=CHECKPOINT, sku_mapping=MAPPING,
)
if not validation_query.awaitTermination(60):
    raise TimeoutError("테스트 쿼리가 아직 실행 중입니다. status/exception을 확인하세요")
assert counts() == {"silver": EXPECTED["v1_v2_replay"]["silver_events"], "quarantine": 0}
expected_common = json.loads((FIXTURES / "expected_common_events.json").read_text(encoding="utf-8"))
expected_frame = (spark.createDataFrame([(json.dumps(row),) for row in expected_common], ["common_json"])
                  .select(F.from_json("common_json", pipeline.COMMON_SCHEMA,
                                      {"mode": "FAILFAST", "timeZone": "UTC"}).alias("common"))
                  .select("common.*"))
actual_frame = spark.table(SILVER).select(*pipeline.COMMON_SCHEMA.fieldNames())
assert actual_frame.exceptAll(expected_frame).count() == 0
assert expected_frame.exceptAll(actual_frame).count() == 0
print(counts())


## 4. 동일 입력·오류 입력 재시도

- 같은 batch ID의 실제 입력을 다시 저장해도 행 수가 증가하지 않아야 합니다.
- 오류 16개는 각 독립 기대 rule_id/error_code와 일치해야 합니다.

In [ ]:
pipeline.write_batch(bronze_rows(replay), 0, sku_mapping=MAPPING,
                     silver_table=SILVER, quarantine_table=QUARANTINE)
assert counts() == {"silver": 6, "quarantine": 0}
errors = bronze_rows([case["raw_payload"] for case in ERROR_CASES], 100)
for attempt in range(2):
    pipeline.write_batch(errors, 1, sku_mapping=MAPPING,
                         silver_table=SILVER, quarantine_table=QUARANTINE)
assert counts() == {"silver": 6, "quarantine": EXPECTED["error_cases"]["quarantine_records"]}
actual_codes = sorted((row.rule_id, row.error_code) for row in
                      spark.table(QUARANTINE).select("rule_id", "error_code").collect())
assert actual_codes == sorted((case["rule_id"], case["error_code"]) for case in ERROR_CASES)
print(counts())


## 5. 같은 키의 다른 업무 값

- 새 키가 같은 batch에서 충돌하면 양쪽 모두 격리합니다.
- 이미 저장된 키가 충돌하면 기존 값은 유지하고 새 값만 격리합니다.

In [ ]:
new_conflicts = (FIXTURES / "conflict_events.jsonl").read_text(encoding="utf-8").splitlines()
pipeline.write_batch(bronze_rows(new_conflicts, 200), 2, sku_mapping=MAPPING,
                     silver_table=SILVER, quarantine_table=QUARANTINE)
assert counts() == {"silver": 6, "quarantine": 18}
changed = json.loads(replay[0])
changed["payload"]["total"] = 50001
pipeline.write_batch(bronze_rows([json.dumps(changed)], 300), 3, sku_mapping=MAPPING,
                     silver_table=SILVER, quarantine_table=QUARANTINE)
assert counts() == {"silver": 6, "quarantine": 19}
assert spark.table(SILVER).where("event_id = 'a-poc-v1-01'").first().data.order_amount == 50000


## 6. Quarantine 커밋 후 Silver 실패·재시도

- 테스트에서 Silver 쓰기만 한 번 실패시킵니다. 운영 함수는 오류를 숨기지 않습니다.
- 재시도 시 Quarantine은 중복되지 않고 미완료 Silver 쓰기가 수행되어야 합니다. 두 커밋의 원자성을 의미하지 않습니다.

In [ ]:
from unittest.mock import patch

retry_good = json.loads(replay[0])
retry_good["event_id"] = "a-retry-good"
retry_bad = {**retry_good, "event_id": "a-retry-bad"}
retry_bad.pop("order_no")
retry_input = bronze_rows([json.dumps(retry_good), json.dumps(retry_bad)], 400)
original_merge = pipeline.merge_insert_only

def fail_silver_once(rows, table, keys):
    if table == SILVER:
        raise RuntimeError("INJECTED_SILVER_WRITE_FAILURE")
    return original_merge(rows, table, keys)

try:
    with patch.object(pipeline, "merge_insert_only", side_effect=fail_silver_once):
        pipeline.write_batch(retry_input, 4, sku_mapping=MAPPING,
                             silver_table=SILVER, quarantine_table=QUARANTINE)
except RuntimeError as error:
    if str(error) != "INJECTED_SILVER_WRITE_FAILURE":
        raise
else:
    raise AssertionError("의도한 저장 실패가 발생하지 않았습니다")
assert counts() == {"silver": 6, "quarantine": 20}
for attempt in range(2):
    pipeline.write_batch(retry_input, 4, sku_mapping=MAPPING,
                         silver_table=SILVER, quarantine_table=QUARANTINE)
assert counts() == {"silver": 7, "quarantine": 20}


## 7. 같은 체크포인트 재시작·새 전달

- 원본 12건을 새 offset으로 재전달합니다. 같은 오류에 새 전송 메타데이터를 붙여 함께 전달합니다.
- 재시작 후 정상/격리 수가 유지되어야 합니다.

In [ ]:
invalid_replay = json.loads(next(case["raw_payload"] for case in ERROR_CASES if case["case_id"] == "unmapped_sku"))
invalid_replay.update(test_run_id="new-delivery", published_at="2026-10-12T12:00:00Z")
bronze_rows([*replay, json.dumps(invalid_replay)], 500).write.format("delta").mode("append").saveAsTable(BRONZE)
validation_query = pipeline.start_stream(
    spark, bronze_table=BRONZE, silver_table=SILVER, quarantine_table=QUARANTINE,
    checkpoint=CHECKPOINT, sku_mapping=MAPPING,
)
if not validation_query.awaitTermination(60):
    raise TimeoutError("재시작 쿼리가 아직 실행 중입니다")
assert counts() == {"silver": 7, "quarantine": 20}
print({"run_id": RUN_ID, "result": "PASS", **counts()})
display(spark.table(QUARANTINE).groupBy("rule_id", "error_code").count())


## 실행 근거와 한계

- 모든 셀이 통과한 뒤 실행 ID·테이블 이름·최종 결과를 보관합니다.
- 작은 합성 입력과 단일 writer의 검증입니다. 다중 writer·대량 처리·Gold 정확성을 입증하지 않습니다.
- 실패 후 처음부터 실행하면 새 실행 ID로 별도 시험을 만듭니다.